# MCP HTTP Research Notebook

This notebook validates Meridian's MCP server over HTTP before any backend or agent work.

It covers:
- connectivity via `initialize`
- tool discovery via `tools/list`
- one product-related tool call
- one order-related tool call


In [1]:
import asyncio
import json
import os
from dataclasses import dataclass
from json import JSONDecodeError
from typing import Any

import httpx

In [2]:
MCP_SERVER_URL = os.getenv(
    "MCP_SERVER_URL",
    "https://order-mcp-74afyau24q-uc.a.run.app/mcp",
)


@dataclass
class JsonRpcResult:
    method: str
    request: dict[str, Any]
    response: dict[str, Any] | str


class McpHttpProbe:
    def __init__(self, url: str) -> None:
        self.url = url
        self.request_id = 0
        self.session_id: str | None = None

    def _next_id(self) -> int:
        self.request_id += 1
        return self.request_id

    async def post(
        self,
        method: str,
        params: dict[str, Any] | None = None,
        *,
        include_session: bool = True,
    ) -> JsonRpcResult:
        payload: dict[str, Any] = {
            "jsonrpc": "2.0",
            "id": self._next_id(),
            "method": method,
        }
        if params is not None:
            payload["params"] = params

        headers = {
            "accept": "application/json, text/event-stream",
            "content-type": "application/json",
        }
        if include_session and self.session_id:
            headers["mcp-session-id"] = self.session_id

        async with httpx.AsyncClient(timeout=30.0) as client:
            response = await client.post(self.url, json=payload, headers=headers)
            response.raise_for_status()
            self.session_id = response.headers.get("mcp-session-id", self.session_id)
            content_type = response.headers.get("content-type", "")
            if not response.content:
                body = ""
            elif "application/json" in content_type:
                body = response.json()
            else:
                body = response.text
            return JsonRpcResult(method=method, request=payload, response=body)

    async def notify(
        self,
        method: str,
        params: dict[str, Any] | None = None,
    ) -> JsonRpcResult:
        payload: dict[str, Any] = {"jsonrpc": "2.0", "method": method}
        if params is not None:
            payload["params"] = params

        headers = {
            "accept": "application/json, text/event-stream",
            "content-type": "application/json",
        }
        if self.session_id:
            headers["mcp-session-id"] = self.session_id

        async with httpx.AsyncClient(timeout=30.0) as client:
            response = await client.post(self.url, json=payload, headers=headers)
            response.raise_for_status()
            self.session_id = response.headers.get("mcp-session-id", self.session_id)
            content_type = response.headers.get("content-type", "")
            if not response.content:
                body = ""
            elif "application/json" in content_type:
                try:
                    body = response.json()
                except JSONDecodeError:
                    body = response.text
            else:
                body = response.text
            return JsonRpcResult(method=method, request=payload, response=body)


def pretty_print(title: str, data: Any) -> None:
    print(f"\n=== {title} ===")
    if isinstance(data, str):
        print(data)
        return
    print(json.dumps(data, indent=2, sort_keys=True, default=str))


In [3]:
async def run_mcp_probe() -> None:
    probe = McpHttpProbe(MCP_SERVER_URL)

    print(f"MCP_SERVER_URL={MCP_SERVER_URL}")

    initialize = await probe.post(
        "initialize",
        {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "andela-assessment-probe", "version": "0.1.0"},
        },
        include_session=False,
    )
    pretty_print("initialize request", initialize.request)
    pretty_print("initialize response", initialize.response)

    initialized = await probe.notify("notifications/initialized", {})
    pretty_print("notifications/initialized request", initialized.request)
    pretty_print("notifications/initialized response", initialized.response)

    tools_list = await probe.post("tools/list", {})
    pretty_print("tools/list request", tools_list.request)
    pretty_print("tools/list response", tools_list.response)

    for tool_name, arguments in [("list_products", {}), ("list_orders", {})]:
        result = await probe.post(
            "tools/call",
            {"name": tool_name, "arguments": arguments},
        )
        pretty_print(f"tools/call request for {tool_name}", result.request)
        pretty_print(f"tools/call response for {tool_name}", result.response)


In [4]:
await run_mcp_probe()

MCP_SERVER_URL=https://order-mcp-74afyau24q-uc.a.run.app/mcp

=== initialize request ===
{
  "id": 1,
  "jsonrpc": "2.0",
  "method": "initialize",
  "params": {
    "capabilities": {},
    "clientInfo": {
      "name": "andela-assessment-probe",
      "version": "0.1.0"
    },
    "protocolVersion": "2025-03-26"
  }
}

=== initialize response ===
{
  "id": 1,
  "jsonrpc": "2.0",
  "result": {
    "capabilities": {
      "experimental": {},
      "prompts": {
        "listChanged": false
      },
      "resources": {
        "listChanged": false,
        "subscribe": false
      },
      "tools": {
        "listChanged": false
      }
    },
    "protocolVersion": "2025-03-26",
    "serverInfo": {
      "name": "order-mcp",
      "version": "1.22.0"
    }
  }
}

=== notifications/initialized request ===
{
  "jsonrpc": "2.0",
  "method": "notifications/initialized",
  "params": {}
}

=== notifications/initialized response ===


=== tools/list request ===
{
  "id": 2,
  "jsonrpc": "2.0",